# Depth Anything サンプル

## Depth Anything — 単眼深度推定

このノートブックは **Depth Anything V2** を使った単眼深度推定のサンプルです。

```
画像（URL またはファイル）→ Depth Anything モデル → 深度マップ（グレースケール / カラーマップ）
```

単眼深度推定は、1枚の RGB 画像から**ピクセルごとの深度マップ**を予測します。ステレオカメラや LiDAR は不要です。  
出力の深度マップでは、近い物体が明るく（または暖色で）、遠い物体が暗く（または寒色で）表示されます。

### 使用可能なモデル

| モデル | パラメータ数 | 速度 |
|-------|------------|------|
| `depth-anything/Depth-Anything-V2-Large-hf` | 335 M | 最も遅い / 最も高精度 |
| `depth-anything/Depth-Anything-V2-Base-hf`  |  97 M | — |
| `depth-anything/Depth-Anything-V2-Small-hf` |  24 M | 最も速い / 最も軽量 |

writefile セル内の `MODEL_NAME` を変更することでモデルを切り替えられます。

📄 [Depth Anything V2 論文](https://arxiv.org/abs/2406.09414)  
🤗 [Hugging Face モデルハブ](https://huggingface.co/depth-anything)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = "/content/drive/MyDrive/CV-Samples/depth"
!mkdir -p "{PROJECT_PATH}"
%cd "{PROJECT_PATH}"
!mkdir -p output
!ls


In [ ]:
# Install dependencies
!pip install transformers accelerate -q

import transformers
print(transformers.__version__)  # check version


In [ ]:
# Download sample images
import os

# Image 1: repair scene (from Pixabay CDN)
REPAIR_URL = "https://cdn.pixabay.com/photo/2019/11/04/20/06/repair-4602070_640.jpg"
repair_dest = f'{PROJECT_PATH}/repair-4602070_640.jpg'
if not os.path.exists(repair_dest):
    os.system(f'wget -q "{REPAIR_URL}" -O "{repair_dest}"')
    print('Downloaded: repair-4602070_640.jpg')
else:
    print('Already exists: repair-4602070_640.jpg')

# Image 2: avenue / trees perspective (from GitHub)
BASE_URL = "https://raw.githubusercontent.com/mastnk/cv-samples/main/depth"
IMAGE_FILES = [
    "bertvthul-avenue-815297_640.jpg",
]
for fname in IMAGE_FILES:
    url  = f'{BASE_URL}/{fname}'
    dest = f'{PROJECT_PATH}/{fname}'
    if not os.path.exists(dest):
        os.system(f'wget -q "{url}" -O "{dest}"')
        print(f'Downloaded: {fname}')
    else:
        print(f'Already exists: {fname}')

%cd "{PROJECT_PATH}"
!ls


## 独自画像を使うには

画像を用意する方法は 2 つあります。

**① URL を指定する**  
スクリプト実行時に `--url` フラグに直接画像 URL を渡します：
```
%run depth_anything.py --url https://cdn.pixabay.com/photo/2019/11/04/20/06/repair-4602070_640.jpg
```

**② 画像ファイルを `PROJECT_PATH/` に置く**  
画像ファイルを `PROJECT_PATH/` に配置してから `--file` または `--dir` で実行します。

アップロードは **Google Drive** 経由が簡単です：  
[drive.google.com](https://drive.google.com) を開き、`マイドライブ / CV-Samples / depth/` に移動してファイルをドラッグ＆ドロップするだけ。  
マウント済みの Drive を通じて Colab から即座にアクセスできます。追加のアップロード手順は不要です。

```
%run depth_anything.py --file your_image.jpg   # ファイル 1 枚
%run depth_anything.py --dir  .                # フォルダ内の全画像
```

## モデルを選択するには

下の writefile セル内の `MODEL_NAME` を変更してモデルを切り替えます。  
`MODEL_NAME = ...` の行が複数ある場合、**有効になるのは最後の 1 行だけ**です。

```python
# MODEL_NAME = 'depth-anything/Depth-Anything-V2-Large-hf'  # 335M params — most accurate, slowest
# MODEL_NAME = 'depth-anything/Depth-Anything-V2-Base-hf'   #  97M params
MODEL_NAME   = 'depth-anything/Depth-Anything-V2-Small-hf'  #  24M params — fastest, lightest  ← active
```

モデルが大きいほど深度マップの精度は高くなりますが、実行時間も長くなります。  
まずは `Depth-Anything-V2-Small-hf` で試してみましょう。

In [ ]:
# Save this cell as a Python file (Execute after editing)
%%writefile depth_anything.py
"""Depth Anything V2 — command-line interface.

Usage:
  %run depth_anything.py --url  <image_url>  --out <output_path> [--disp] [--cmap COLORMAP]
  %run depth_anything.py --file <image_path> --out <output_path> [--disp] [--cmap COLORMAP]
  %run depth_anything.py --dir  <image_dir>  --out <output_dir>  [--disp] [--cmap COLORMAP]
"""

import argparse
import glob
import os

import numpy as np
from PIL import Image
import requests
from io import BytesIO
import matplotlib.cm as cm
from transformers import pipeline

# ---- IPython display (works when executed via %run in Colab) -----
try:
    from IPython.display import display as ipy_display
    _has_ipy = True
except ImportError:
    _has_ipy = False

# ---- Configuration -----------------------------------------------
MODEL_NAME = 'depth-anything/Depth-Anything-V2-Small-hf'  # fastest / lightest
# MODEL_NAME = 'depth-anything/Depth-Anything-V2-Base-hf'   #  97M params
# MODEL_NAME = 'depth-anything/Depth-Anything-V2-Large-hf'  # most accurate

PROJECT_PATH = '/content/drive/MyDrive/CV-Samples/depth'

# ---- Model loading -----------------------------------------------
pipe = pipeline(
    task='depth-estimation',
    model=MODEL_NAME,
    device=0 if __import__('torch').cuda.is_available() else -1,
)

# ---- Depth map helper --------------------------------------------
def depth_to_image(depth_array: np.ndarray, cmap: str = 'inferno') -> Image.Image:
    """Convert a raw depth array to a colorized PIL image."""
    d = depth_array.astype(np.float32)
    d = (d - d.min()) / (d.max() - d.min() + 1e-8)  # normalize 0-1
    colored = (cm.get_cmap(cmap)(d)[:, :, :3] * 255).astype(np.uint8)
    return Image.fromarray(colored)

# ---- Display helper ----------------------------------------------
def show(original: Image.Image, depth_img: Image.Image, label: str, disp: bool) -> None:
    """When --disp is active, display original and depth map side by side."""
    if disp:
        w, h = original.size
        combined = Image.new('RGB', (w * 2, h))
        combined.paste(original, (0, 0))
        combined.paste(depth_img.resize((w, h)), (w, 0))
        if _has_ipy:
            ipy_display(combined)
        print(label)

# ---- Functions ---------------------------------------------------
def estimate_url(url: str, out: str, cmap: str = 'inferno', disp: bool = False):
    """Download an image from a URL and estimate depth."""
    image     = Image.open(BytesIO(requests.get(url).content)).convert('RGB')
    result    = pipe(image)
    depth_img = depth_to_image(np.array(result['depth']), cmap)
    show(image, depth_img, url, disp)
    os.makedirs(os.path.dirname(os.path.abspath(out)), exist_ok=True)
    depth_img.save(out)
    print(f'Saved: {out}')
    return result


def estimate_file(path: str, out: str, cmap: str = 'inferno', disp: bool = False):
    """Estimate depth in a single local image file."""
    image     = Image.open(path).convert('RGB')
    result    = pipe(image)
    depth_img = depth_to_image(np.array(result['depth']), cmap)
    show(image, depth_img, path, disp)
    os.makedirs(os.path.dirname(os.path.abspath(out)), exist_ok=True)
    depth_img.save(out)
    print(f'Saved: {out}')
    return result


def estimate_dir(directory: str, out_dir: str, cmap: str = 'inferno', disp: bool = False):
    """Estimate depth in all images in a directory."""
    exts = ['jpg', 'jpeg', 'png', 'bmp', 'JPG', 'JPEG', 'PNG', 'BMP']
    filepaths = []
    for ext in exts:
        filepaths += sorted(glob.glob(os.path.join(directory, f'*.{ext}')))

    if not filepaths:
        print(f'No images found in: {directory}')
        return

    os.makedirs(out_dir, exist_ok=True)
    for path in filepaths:
        base      = os.path.splitext(os.path.basename(path))[0]
        out_path  = os.path.join(out_dir, f'depth_{base}.png')
        estimate_file(path, out_path, cmap, disp)
        print()


# ---- Argument parsing --------------------------------------------
parser = argparse.ArgumentParser(description='Depth Anything V2')
group  = parser.add_mutually_exclusive_group(required=True)
group.add_argument('--url',  type=str, help='Image URL for depth estimation')
group.add_argument('--file', type=str, help='Path to a single image file')
group.add_argument('--dir',  type=str, help='Directory of images')
parser.add_argument('--out',  type=str, required=True,
                    help='Output file path (--url/--file) or output directory (--dir)')
parser.add_argument('--cmap', type=str, default='inferno',
                    help='Matplotlib colormap for depth map (default: inferno)')
parser.add_argument('--disp', action='store_true',
                    help='Display original + depth map side by side')
args = parser.parse_args()

# ---- Run ---------------------------------------------------------
if args.url:
    estimate_url(args.url,   out=args.out, cmap=args.cmap, disp=args.disp)
elif args.file:
    estimate_file(args.file, out=args.out, cmap=args.cmap, disp=args.disp)
elif args.dir:
    estimate_dir(args.dir,   out_dir=args.out, cmap=args.cmap, disp=args.disp)


## `depth_anything.py` の使い方

上の `%%writefile` セルを実行すると、`depth_anything.py` が `PROJECT_PATH` に保存されます。  
`--disp` でのインライン画像表示を有効にするため、`!python` ではなく **`%run`** で実行してください。

```
%run depth_anything.py --url  <画像URL>  --out <出力ファイルパス>   # リモート画像の深度推定
%run depth_anything.py --file <画像パス> --out <出力ファイルパス>   # ローカルファイル 1 枚の深度推定
%run depth_anything.py --dir  <ディレクトリ> --out <出力フォルダ>  # フォルダ内の全画像を深度推定
```

**必須引数**

| フラグ | 説明 |
|--------|------|
| `--out <パス>` | 出力ファイルパス（`--url` / `--file`）または出力フォルダ（`--dir`） |

**オプション引数**

| フラグ | デフォルト | 説明 |
|--------|-----------|------|
| `--disp` | オフ | 元画像と深度マップを並べて表示 |
| `--cmap <name>` | `inferno` | 深度マップの Matplotlib カラーマップ |

**カラーマップの選択肢**（`--cmap` で変更）

| カラーマップ | 見た目 |
|------------|--------|
| `inferno` | 黒 → オレンジ → 白（デフォルト） |
| `magma`   | 黒 → 紫 → 白 |
| `plasma`  | 紫 → オレンジ → 黄 |
| `viridis` | 濃い青 → 緑 → 黄 |
| `gray`    | 黒 → 白（クラシック） |

**実行例**

```bash
# URL から深度推定して output/ に保存
%run depth_anything.py --url https://cdn.pixabay.com/photo/2019/11/04/20/06/repair-4602070_640.jpg --out output/depth_repair.png --disp

# ローカルファイルをグレースケールで深度推定
%run depth_anything.py --file bertvthul-avenue-815297_640.jpg --out output/depth_avenue.png --cmap gray --disp

# フォルダ内の全画像を深度推定して output/ に保存
%run depth_anything.py --dir . --out output/ --disp
```

## 実行方法

`depth_anything.py` は `!python` ではなく **`%run`** で実行してください。Colab カーネル内で実行されるため、`--disp` でのインライン画像表示が有効になります。

| モード | フラグ | `--out` |
|--------|--------|---------|
| URL から | `--url <url>` | 出力**ファイル**パス |
| ファイル 1 枚 | `--file <パス>` | 出力**ファイル**パス |
| ディレクトリ | `--dir <パス>` | 出力**フォルダ**パス |

`--out` は全モードで**必須**です。  
`--disp` を追加すると、元画像と深度マップが並んで表示されます。  
`--cmap <name>` でカラーマップを変更できます（デフォルト: `inferno`）。

In [ ]:
# Execute: estimate depth from URL (with display)
%cd "{PROJECT_PATH}"
%run depth_anything.py --disp --url https://cdn.pixabay.com/photo/2019/11/04/20/06/repair-4602070_640.jpg --out output/depth_repair.png


In [ ]:
# Execute: estimate depth from a single local image file (with display)
%cd "{PROJECT_PATH}"
%run depth_anything.py --disp --file bertvthul-avenue-815297_640.jpg --out output/depth_avenue.png


In [ ]:
# Execute: estimate depth for all images in a directory (with display)
%cd "{PROJECT_PATH}"
%run depth_anything.py --disp --dir . --out output/


💾 **ノートブックを保存するのを忘れずに！**

作業内容を保持するため、閉じる前に **ファイル → ドライブにコピーを保存** を実行してください。